# Suite SCH — Schemas

Builders, `Spec` resolution, native conversion, and on-demand `validate()` for every schema kind.
See `TestPlan.md` for the plan. Each cell is one test case, named by its ID.

In [ ]:
import base64
import math

from mbse_schemas.Framework import Schemas as S
from mbse_schemas.Framework.Errors import DecodeError
from support import raises, text

NATIVES = (int, float, str, bool, bytes)

## SCH-01 · `OfNative.Data` equality and validation
Supported natives validate; unsupported types and a missing type are reported. `bool` is not `int`.

In [ ]:
for native in NATIVES:
    assert S.OfNative.Data(native).validate() == [], native
for unsupported in (complex, list, dict, tuple, bytearray, type(None), object, None):
    assert S.OfNative.Data(unsupported).validate(), unsupported
assert S.OfNative.Data(str) == S.OfNative.Data(str)
assert S.OfNative.Data(bool) != S.OfNative.Data(int)
assert S.OfNative.Builder().create() == S.OfNative.Data(None)
assert S.OfNative.Builder().create().validate()

## SCH-02 · `OfNative.resolve` accepts a type, a callable, or data

In [ ]:
assert S.OfNative.resolve(str) == S.OfNative.Data(str)
assert S.OfNative.resolve(lambda n: n.type(bytes)) == S.OfNative.Data(bytes)
existing = S.OfNative.Data(int)
assert S.OfNative.resolve(existing) is existing
with raises(TypeError, match="must return its builder"):
    S.OfNative.resolve(lambda n: None)
with raises(TypeError):
    S.OfNative.resolve("str")
with raises(TypeError):
    S.OfNative.resolve(3)

## SCH-03 · `to_plain` / `from_plain`: exact types, both directions
Distinct native types are never coerced; subclasses of natives are rejected too. Writing a wrong type is a program
error (`TypeError`); reading one is a problem in the data (`DecodeError`, a `ValueError`, with no location here).

In [ ]:
cases = {int: [0, -1, 2**100, -(2**63)], float: [0.0, -0.0, 0.1, 1e308, 5e-324], str: ["", "x", "日本語 🚀", "\x00"],
         bool: [True, False], bytes: [b"", b"\x00\xff", bytes(range(256))]}
for native, values in cases.items():
    schema = S.OfNative.Data(native)
    for value in values:
        back = schema.from_plain(schema.to_plain(value))
        assert type(back) is native and back == value, (native, value)
        if native is float:
            assert math.copysign(1, back) == math.copysign(1, value)

wrong = {int: [True, 1.0, "1", None], float: [1, True, None], str: [b"x", 1, None], bool: [0, 1, "true", None],
         bytes: [bytearray(b"x"), "AA==", None]}
for native, values in wrong.items():
    for value in values:
        with raises(TypeError):
            S.OfNative.Data(native).to_plain(value)


class MyInt(int):
    pass


with raises(TypeError):
    S.OfNative.Data(int).to_plain(MyInt(3))
with raises(DecodeError, match="expected int, got MyInt"):
    S.OfNative.Data(int).from_plain(MyInt(3))

for native, plain, message in [
    (int, True, "expected int, got bool"),
    (int, 1.0, "expected int, got float"),
    (float, 1, "expected float, got int"),
    (bool, 0, "expected bool, got int"),
    (str, b"x", "expected str, got bytes"),
    (str, None, "expected str, got NoneType"),
    (bytes, b"raw", "expected base64 text for bytes, got bytes"),
]:
    try:
        S.OfNative.Data(native).from_plain(plain)
    except DecodeError as error:
        assert str(error) == error.reason == message and (error.line, error.column, error.path) == (None, None, None)
    else:
        raise AssertionError((native, plain))

## SCH-04 · Bytes are strict base64 text

In [ ]:
schema = S.OfNative.Data(bytes)
assert schema.to_plain(b"\x00\xff") == "AP8="
assert schema.from_plain(base64.b64encode(b"hello").decode()) == b"hello"
for bad in ["AP8", "AP8=\n", "A P8=", "not base64!", "AP8=="]:
    with raises(DecodeError, match="invalid base64 text"):
        schema.from_plain(bad)

## SCH-05 · Non-finite floats have exactly three plain forms

In [ ]:
schema = S.OfNative.Data(float)
assert schema.to_plain(math.inf) == "Infinity" and schema.to_plain(-math.inf) == "-Infinity"
assert schema.to_plain(math.nan) == "NaN" and schema.to_plain(float("-nan")) == "NaN"
assert schema.from_plain("Infinity") == math.inf and schema.from_plain("-Infinity") == -math.inf
assert math.isnan(schema.from_plain("NaN"))
for bad in ["nan", "inf", "+Infinity", "infinity", "NAN", "1.5", ""]:
    with raises(DecodeError, match=f"expected a float or one of ['-Infinity', 'Infinity', 'NaN'], got {bad!r}"):
        schema.from_plain(bad)
with raises(DecodeError, match="expected str, got float"):
    S.OfNative.Data(str).from_plain(1.5)  # strings are only special for float

## SCH-06 · `OfAny.Builder`: selecting a kind yields that kind's data

In [ ]:
assert type(S.OfAny.Builder().as_native(str).create()) is S.OfNative.Data
obj = S.OfAny.Builder().as_object(lambda o: o.properties(text("x"))).create()
assert type(obj) is S.OfObject.Data and list(obj.properties) == ["x"]
assert S.OfAny.Builder().as_native(str).as_native(int).create() == S.OfNative.Data(int)  # the last selection wins
with raises(ValueError, match="no kind selected"):
    S.OfAny.Builder().create()

## SCH-07 · `OfAny.Builder` with a source: create / clone / update

In [ ]:
source = S.OfNative.Data(str)
with raises(ValueError):
    S.OfAny.Builder(source).as_native(int).create()
copy_ = S.OfAny.Builder(source).clone()
assert copy_ == source and copy_ is not source
assert S.OfAny.Builder(source).as_native(int).clone() == S.OfNative.Data(int) and source == S.OfNative.Data(str)
assert S.OfAny.Builder(source).as_native(bytes).update() is source and source.type is bytes
with raises(TypeError, match="cannot change the kind"):
    S.OfAny.Builder(source).as_object(lambda o: o).update()
with raises(ValueError):
    S.OfAny.Builder(source).update()  # nothing selected
with raises(ValueError):
    S.OfAny.Builder().clone()
with raises(ValueError):
    S.OfAny.Builder().as_native(str).update()

## SCH-08 · `OfAny.resolve` accepts value kinds only

In [ ]:
for data in [S.OfNative.Data(str), S.OfObject.Data(), S.OfUnion.Data(), S.OfIntersection.Data()]:
    assert S.OfAny.resolve(data) is data
relation = S.OfRelation.Builder().links("a", "b").create()
with raises(TypeError):
    S.OfAny.resolve(relation)
with raises(TypeError):
    S.OfObject.Builder().properties(lambda p: p.name("r").of(relation)).create()
with raises(TypeError):
    S.OfAny.resolve(42)

## SCH-09 · Finalization rules hold for every builder

In [ ]:
builders = [
    (S.OfNative.Builder, S.OfNative.Data(str)),
    (S.OfObject.Builder, S.OfObject.Data()),
    (S.OfRelation.Builder, S.OfRelation.Data()),
    (S.OfAdjacency.Builder, S.OfAdjacency.Data()),
    (S.OfUnion.Builder, S.OfUnion.Data()),
    (S.OfIntersection.Builder, S.OfIntersection.Data()),
    (S.OfProperty.Builder, S.OfProperty.Data()),
]
for Builder, source in builders:
    assert type(Builder().create()) is type(source)
    with raises(ValueError, match="only valid without a source"):
        Builder(source).create()
    with raises(ValueError, match="only valid with a source"):
        Builder().clone()
    with raises(ValueError, match="only valid with a source"):
        Builder().update()
    cloned = Builder(source).clone()
    assert cloned is not source and type(cloned) is type(source)
    assert Builder(source).update() is source

## SCH-10 · Builders copy their own containers, never referenced schemas

In [ ]:
Rel = S.OfRelation.Builder().links("a", "b").create()
Base = S.OfObject.Builder().properties(text("x")).relations(lambda a: a.name("r").of(Rel).me("a")).create()

extended = S.OfObject.Builder(Base).properties(text("y")).clone()
assert list(Base.properties) == ["x"] and list(extended.properties) == ["x", "y"]
assert extended.properties is not Base.properties and extended.adjacencies is not Base.adjacencies
assert extended.adjacencies["r"] is Base.adjacencies["r"]  # adjacency data is shared, not copied
assert extended.adjacencies["r"].relation is Rel  # references keep identity

builder = S.OfObject.Builder(Base).properties(text("z"))
assert "z" not in Base.properties  # nothing reaches the source before update()
first = builder.clone()
builder.properties(text("w"))
second = builder.clone()
assert "w" not in first.properties and "w" in second.properties

same = S.OfObject.Builder(Base).properties(text("v")).update()
assert same is Base and "v" in Base.properties

## SCH-11 · `OfObject`: properties and relations are keyed by name, in order

In [ ]:
o = (S.OfObject.Builder()
     .properties(text("b"), text("a"), text("b", int))
     .properties(text("c"))
     .create())
assert list(o.properties) == ["b", "a", "c"] and o.properties["b"] == S.OfNative.Data(int)
Rel2 = S.OfRelation.Builder().links("x", "y").create()
o2 = S.OfObject.Builder().relations(lambda a: a.name("r").of(Rel).me("a"), lambda a: a.name("r").of(Rel2).me("x")).create()
assert list(o2.adjacencies) == ["r"] and o2.adjacencies["r"].relation is Rel2
assert S.OfObject.Builder().singleton("Global.Name").create().singleton == "Global.Name"
assert S.OfObject.Data().validate() == []

## SCH-12 · `OfObject.validate` reports clashes and nested problems

In [ ]:
clash = S.OfObject.Builder().properties(text("r")).relations(lambda a: a.name("r").of(Rel).me("a")).create()
assert any("both property and adjacency" in p for p in clash.validate())
nested = S.OfObject.Builder().properties(text("bad", list)).create()
assert nested.validate() == ["property 'bad': unsupported native type <class 'list'>"]
wrong_side = S.OfObject.Builder().relations(lambda a: a.name("r").of(Rel).me("c")).create()
assert any("'c' is not a link" in p for p in wrong_side.validate())
no_relation = S.OfObject.Builder().relations(lambda a: a.name("r").me("a")).create()
assert any("has no relation" in p for p in no_relation.validate())

## SCH-13 · `OfRelation.validate` covers link, property and uniqueness problems

In [ ]:
def problems(builder):
    return builder.create().validate()


assert problems(S.OfRelation.Builder().links("a", "b")) == []
assert any("one-link" in p for p in problems(S.OfRelation.Builder().links("a")))
assert any("at least two links" in p for p in problems(S.OfRelation.Builder()))
assert any("duplicate link" in p for p in problems(S.OfRelation.Builder().links("a", "a")))
assert any("both link and property" in p for p in problems(S.OfRelation.Builder().links("a", "b").properties(text("a"))))
assert any("unknown" in p for p in problems(S.OfRelation.Builder().links("a", "b").unique("c")))
assert problems(S.OfRelation.Builder().links("a", "b").unique()) == []  # unique() is trivially true
assert problems(S.OfRelation.Builder().links("a", "b").properties(text("k")).unique("a", "b", "k")) == []
assert any("property 'k'" in p for p in problems(S.OfRelation.Builder().links("a", "b").properties(text("k", list))))
assert S.OfRelation.Builder().links("a", "b").links("c", "d").create().links == ("c", "d")  # links() replaces
r = S.OfRelation.Builder().links("a", "b").unique("a").unique("b").unique("a").create()
assert r.uniques == (frozenset({"a"}), frozenset({"b"}), frozenset({"a"}))  # clauses accumulate as written

## SCH-14 · Unions: branches accumulate in order; validation checks count, kinds and predicates

In [ ]:
A = S.OfObject.Builder().properties(text("a")).create()
B = S.OfObject.Builder().properties(text("b")).create()
u = S.OfUnion.Builder().branches(lambda b: b.of(A).when("p")).branches(lambda b: b.of(B).when("q")).create()
assert [b.type for b in u.branches] == [A, B] and [b.when for b in u.branches] == ["p", "q"]
assert u.validate() == []
assert any("at least two" in p for p in S.OfUnion.Builder().branches(lambda b: b.of(A).when("p")).create().validate())
native = lambda t: t.as_native(str)
assert any("same kind" in p for p in
           S.OfUnion.Builder().branches(lambda b: b.of(A).when("p"), lambda b: b.of(native).when("q")).create().validate())
with raises(TypeError, match="as_native(str)"):
    S.OfUnion.Builder().branches(lambda b: b.of(str))  # a bare class is not an OfAny.Spec
with raises(TypeError, match="must return its builder"):
    S.OfObject.Builder().properties(lambda p: "not a builder")
missing = S.OfUnion.Builder().branches(lambda b: b.of(A), lambda b: b.of(B).when("q")).create().validate()
assert missing == ["branch 0 has no discriminator predicate"]

## SCH-15 · Intersections: parts accumulate; conflicting property types are reported

In [ ]:
X1 = S.OfObject.Builder().properties(text("x")).create()
X2 = S.OfObject.Builder().properties(lambda p: p.name("x").of(lambda t: t.as_native(str)), text("y")).create()
X3 = S.OfObject.Builder().properties(text("x", int)).create()
assert S.OfIntersection.Builder().of(X1).of(X2).create().parts == (X1, X2)
assert S.OfIntersection.Builder().of(X1, X2).create().validate() == []  # equal types, not identical objects
assert any("conflicting" in p for p in S.OfIntersection.Builder().of(X1, X2, X3).create().validate())
assert any("at least two" in p for p in S.OfIntersection.Builder().of(X1).create().validate())
assert any("same kind" in p for p in S.OfIntersection.Builder().of(X1, S.OfNative.Data(str)).create().validate())

## SCH-16 · Finding F2 (pinned): `validate()` accepts names that bindings may reserve
Empty names, names with spaces or `$`, leading underscores, and names of builder methods all validate. Whether the
schema layer should reject them is an open question in `FRAMEWORK.md`.

In [ ]:
odd = S.OfObject.Builder().properties(*(text(n) for n in ["", "with space", "$ref", "_values", "create", "accept"])).create()
assert odd.validate() == []

## SCH-17 · Union and intersection selection; identity equality; Spec errors name what they got

In [ ]:
U = S.OfAny.Builder().as_union(lambda u: u.branches(lambda b: b.of(A).when("p"), lambda b: b.of(B).when("q"))).create()
I = S.OfAny.Builder().as_intersection(lambda i: i.of(X1, X2)).create()
assert type(U) is S.OfUnion.Data and len(U.branches) == 2 and type(I) is S.OfIntersection.Data and I.parts == (X1, X2)
assert S.OfUnion.resolve(U) is U and S.OfIntersection.resolve(I) is I
assert type(S.OfUnion.resolve(lambda u: u)) is S.OfUnion.Data and type(S.OfIntersection.resolve(lambda i: i)) is S.OfIntersection.Data
for data in [S.OfObject.Data(), S.OfRelation.Data(), S.OfAdjacency.Data(), S.OfUnion.Data(), S.OfIntersection.Data(),
             S.OfProperty.Data()]:
    assert data == data and data != type(data)()  # schemas other than natives compare by identity
assert S.OfObject.Data(properties={"x": "junk"}).validate() == ["property 'x': not a schema: 'junk'"]
untyped = S.OfUnion.Builder().branches(lambda b: b.when("p"), lambda b: b.of(A).when("q")).create()
assert untyped.branches[0].type is None and untyped.branches[0] == untyped.branches[0]
assert "union branches must all be the same kind" in untyped.validate()


class Custom:
    pass


with raises(TypeError, match="a class is not a Spec here"):
    S.OfObject.resolve(Custom)
with raises(TypeError, match="must return its builder, got 'not a builder'"):
    S.OfObject.Builder().properties(lambda p: "not a builder")
with raises(TypeError, match="must return its builder, got <object object"):
    S.OfObject.Builder().properties(lambda p: object())
with raises(TypeError, match="must return its builder, got <function"):
    S.OfObject.Builder().properties(lambda p: (lambda: None))